# Text CLIP Fine-Tuned Model Comparison Example
This notebook shows how to load fine-tuned model checkpoints to evaluate the effectiveness of the fine-tuned weights vs. the base model weights.

In [ ]:
import os
import sys

if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

from torchvision.transforms.v2 import RandomChoice, RandomResizedCrop

from examples.example_scenes import (
    BlenderManScene,
    CandleScene,
    CarScene,
    CarStudioScene,
    DinoScene,
    EinarScene,
    EinarSmallDomeScene,
    FlowerPotScene,
    HouseScene,
    RedCarScene,
    SciFiRobotScene,
    SpringPortraitScene,
    SpringPortraitSmallDomeScene,
    SpringScene,
)
from losses.clip_like import CLIPDirectionalCosineSimilarity
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook
from utils.model.model_utils import create_clip_model_and_tokenizer
from utils.optimize import optimize_with_criterion


In [ ]:
# Select the scene to optimize (uncomment the desired scene)
scene = SciFiRobotScene(device=device)
# scene = SpringScene(device=device)
# scene = CarScene(device=device)
# scene = BlenderManScene(device=device)
# scene = RedCarScene(device=device)
# scene = CandleScene(device=device)
# scene = HouseScene(device=device)
# scene = DinoScene(device=device)
# scene = FlowerPotScene(device=device)
# scene = CarStudioScene(configuration='dome_lights', device=device)
# scene = EinarScene(device=device)
# scene = EinarSmallDomeScene(device=device)
# scene = SpringPortraitScene(device=device)
# scene = SpringPortraitSmallDomeScene(device=device)


In [ ]:
# Hyperparameters
lr = 0.05
n_iter = 250
global_seed = 3

clip_model_name = "ViT-B-16-SigLIP-512"
clip_pretrained = "webli"
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())

models_to_test = [
    None,  # Standard pre-trained model
    "siglip_blend-training-data_64-output-dim.pt",  # Fine-tuned model checkpoint, which will be pulled from HuggingFace if not present locally
]

initial_text = "flat, unappealing lighting"
target_text = "golden hour sunset, warm glow"

for fine_tune in models_to_test:
    print(f"--- Running optimization with fine_tune={fine_tune} ---")
    model, tokenizer, preprocess_eval = create_clip_model_and_tokenizer(
        clip_model_name,
        device=device,
        pretrained=clip_pretrained,
        fine_tune=fine_tune,
    )

    criterion = CLIPDirectionalCosineSimilarity(
        initial_text,
        target_text,
        scene.get_combined_image(color_space_converter).permute(2, 1, 0),
        model,
        tokenizer,
        device=device,
        preprocess=preprocess_eval,
    )

    model_label = "Fine-Tuned Model" if fine_tune else "Original Model"
    title_prefix = f"CLIP Comparison ({model_label})"

    size = model.visual.preprocess_cfg["size"] or (224, 224) # type: ignore

    optimize_with_criterion(
        scene,
        lr,
        n_iter,
        criterion,
        starting_multiplier_std=(0.3, 0.3, 0.3),
        output_subdirectory_name="text_clip_finetune_example",
        n_results=4,
        augmentation=RandomChoice([RandomResizedCrop(size=size, scale=(0.1, 1.0), antialias=True)]), # type: ignore
        render_color_space_converter=color_space_converter,
        require_physically_plausible_multipliers=True,
        title_prefix=title_prefix,
        device=device,
        save_every=25,
        model_name=clip_model_name,
        pretrained_source=fine_tune,
        seed=global_seed,
        show_images_after_augmentation=False,
    )
